In [2]:
!pip install requests beautifulsoup4 pandas

In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [4]:
# TASK 1 — Scrape Data

books = []

categories = [
    "travel_2",
    "mystery_3",
    "historical-fiction_4"
]

BASE_URL = "https://books.toscrape.com/"

for category in categories:

    page = 1

    while True:

        if page == 1:
            url = f"{BASE_URL}catalogue/category/books/{category}/index.html"
        else:
            url = f"{BASE_URL}catalogue/category/books/{category}/page-{page}.html"

        response = requests.get(url)

        if response.status_code != 200:
            break

        soup = BeautifulSoup(response.text, "html.parser")

        books_on_page = soup.select("article.product_pod")

        if len(books_on_page) == 0:
            break

        for book in books_on_page:

            title = book.h3.a["title"]

            price = book.select_one(".price_color").text

            rating = book.p["class"][1]

            availability = (
                book.select_one(".availability")
                .text
                .strip()
            )

            books.append({
                "title": title,
                "price": price,
                "star_rating": rating,
                "availability": availability,
                "category": category
            })

        page += 1


# Create DataFrame AFTER scraping is complete
df = pd.DataFrame(books)

# Verification required for grading
print("Total Books Scraped:", len(df))
print("Number of Categories:", df["category"].nunique())
print("Categories Scraped:")
print(df["category"].unique())

print("\nDataFrame Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 Rows:")
display(df.head())

Total Books Scraped: 69
Number of Categories: 3
Categories Scraped:
['travel_2' 'mystery_3' 'historical-fiction_4']

DataFrame Shape:
(69, 5)

Columns:
['title', 'price', 'star_rating', 'availability', 'category']

First 5 Rows:


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,travel_2
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,travel_2
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,travel_2
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,travel_2
4,Under the Tuscan Sun,Â£37.33,Three,In stock,travel_2


In [6]:
#Task 2- Clean data
df["price_clean"] = (
    df["price"]
    .str.replace(r"[^0-9.]", "", regex=True)
)

df["price_gbp"] = pd.to_numeric(
    df["price_clean"],
    errors="coerce"
)

median_price = df["price_gbp"].median()

df["price_gbp"] = (
    df["price_gbp"]
    .fillna(median_price)
)

df[["price", "price_gbp"]].head()

rating_map = {
    "One":1,
    "Two":2,
    "Three":3,
    "Four":4,
    "Five":5
}

df["rating"] = df["star_rating"].map(rating_map)


df[["star_rating", "rating"]].head()

df["in_stock"] = (
    df["availability"]
    .str.contains("In stock", case=False)
)

df["rating"] = pd.to_numeric(
    df["rating"],
    errors="coerce"
)

median_rating = df["rating"].median()

df["rating"] = (
    df["rating"]
    .fillna(median_rating)
)

df.dropna(
    subset=["title", "category"],
    inplace=True
)

df.info()

df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69 entries, 0 to 68
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         69 non-null     object 
 1   price         69 non-null     object 
 2   star_rating   69 non-null     object 
 3   availability  69 non-null     object 
 4   category      69 non-null     object 
 5   price_clean   69 non-null     object 
 6   price_gbp     69 non-null     float64
 7   rating        69 non-null     int64  
 8   in_stock      69 non-null     bool   
dtypes: bool(1), float64(1), int64(1), object(6)
memory usage: 4.5+ KB


,title,price,star_rating,availability,category,price_clean,price_gbp,rating,in_stock
0,It's Only the Himalayas,Â£45.17,Two,In stock,travel_2,45.17,45.17,2,True
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,travel_2,49.43,49.43,4,True
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,travel_2,48.87,48.87,3,True
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,travel_2,36.94,36.94,2,True
4,Under the Tuscan Sun,Â£37.33,Three,In stock,travel_2,37.33,37.33,3,True


In [7]:
#Task 3 - (GBP → INR conversion).
GBP_TO_INR = 105.50

df["price_inr"] = (
    df["price_gbp"] * GBP_TO_INR
)
df[[
    "price_gbp",
    "price_inr"
]].head()

,price_gbp,price_inr
0,45.17,4765.435
1,49.43,5214.865
2,48.87,5155.785
3,36.94,3897.170
4,37.33,3938.315


In [8]:
#TASK 4 — Create Database
import sqlite3

conn = sqlite3.connect("books.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS categories(
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS books(
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY(category_id)
        REFERENCES categories(category_id)
)
""")



In [9]:
# -----------------------------
# TASK 5 — Insert Data
# -----------------------------

# Insert categories
for category in df["category"].unique():

    cursor.execute("""
    INSERT OR IGNORE INTO categories(
        category_name
    )
    VALUES(?)
    """, (category,))

conn.commit()


# Read categories table
category_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

# Merge category IDs into dataframe
df = df.merge(
    category_df,
    left_on="category",
    right_on="category_name"
)


# Insert books
for _, row in df.iterrows():

    cursor.execute("""
    INSERT INTO books(
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    )
    VALUES(?,?,?,?,?,?)
    """,
    (
        row["title"],
        row["price_gbp"],
        row["price_inr"],
        int(row["rating"]),
        int(row["in_stock"]),
        int(row["category_id"])
    ))

conn.commit()


# -----------------------------
# TASK 6 — SQL QUERIES
# -----------------------------

query1 = """
SELECT title, rating
FROM books
WHERE rating = 5
"""

query2 = """
SELECT title, price_inr
FROM books
ORDER BY price_inr DESC
"""

query3 = """
SELECT *
FROM books
LIMIT 10
"""

query4 = """
SELECT DISTINCT rating
FROM books
"""

query5 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 40
"""

query6 = """
SELECT
    b.title,
    b.rating,
    c.category_name
FROM books b
JOIN categories c
    ON b.category_id = c.category_id
"""


# Execute queries

result1 = pd.read_sql(query1, conn)

result2 = pd.read_sql(query2, conn)

result3 = pd.read_sql(query3, conn)

result4 = pd.read_sql(query4, conn)

result5 = pd.read_sql(query5, conn)

result6 = pd.read_sql(query6, conn)


# Display outputs

print("Query 1: Books with Rating 5")
display(result1.head())

print("Query 2: Highest Priced Books")
display(result2.head())

print("Query 3: First 10 Books")
display(result3.head())

print("Query 4: Distinct Ratings")
display(result4)

print("Query 5: Books Between £20 and £40")
display(result5.head())

print("Query 6: Join Result")
display(result6.head())

Query 1: Books with Rating 5


,title,rating
0,"1,000 Places to See Before You Die",5
1,A Time of Torment (Charlie Parker #14),5
2,What Happened on Beale Street (Secrets of the ...,5
3,The Bachelor Girl's Guide to Murder (Herringfo...,5
4,The Silkworm (Cormoran Strike #2),5


Query 2: Highest Priced Books


,title,price_inr
0,Boar Island (Anna Pigeon #19),6275.140
1,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,6087.350
2,A Year in Provence (Provence #1),6000.840
3,The Past Never Ends,5960.750
4,The Last Painting of Sara de Vos,5860.525


Query 3: First 10 Books


...

,rating
0,2
1,4
2,3
3,1
4,5


Query 5: Books Between £20 and £40


,title,price_gbp
0,Vagabonding: An Uncommon Guide to the Art of L...,36.94
1,Under the Tuscan Sun,37.33
2,The Great Railway Bazaar,30.54
3,The Road to Little Dribbling: Adventures of an...,23.21
4,Neither Here nor There: Travels in Europe,38.95


Query 6: Join Result


,title,rating,category_name
0,It's Only the Himalayas,2,travel_2
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,4,travel_2
2,See America: A Celebration of Our National Par...,3,travel_2
3,Vagabonding: An Uncommon Guide to the Art of L...,2,travel_2
4,Under the Tuscan Sun,3,travel_2


In [10]:
# Read at least two query results

result1 = pd.read_sql(query1, conn)

result2 = pd.read_sql(query2, conn)

# SQL JOIN result

sql_join = pd.read_sql(query6, conn)

# Read tables

books_df = pd.read_sql(
    "SELECT * FROM books",
    conn
)

categories_df = pd.read_sql(
    "SELECT * FROM categories",
    conn
)

# Pandas merge

merge_join = books_df.merge(
    categories_df,
    on="category_id"
)

merge_join = merge_join[
    [
        "title",
        "rating",
        "category_name"
    ]
]

# Display outputs

print("SQL JOIN Result")
display(sql_join.head())

print("Pandas Merge Result")
display(merge_join.head())

# Verify equality

print("Do they match?")
print(sql_join.equals(merge_join))

SQL JOIN Result


,title,rating,category_name
0,It's Only the Himalayas,2,travel_2
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,4,travel_2
2,See America: A Celebration of Our National Par...,3,travel_2
3,Vagabonding: An Uncommon Guide to the Art of L...,2,travel_2
4,Under the Tuscan Sun,3,travel_2


Pandas Merge Result


,title,rating,category_name
0,It's Only the Himalayas,2,travel_2
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,4,travel_2
2,See America: A Celebration of Our National Par...,3,travel_2
3,Vagabonding: An Uncommon Guide to the Art of L...,2,travel_2
4,Under the Tuscan Sun,3,travel_2


Do they match?
True


In [17]:
from google.colab import files

files.download("books.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>